# Earnings Surprise Screener (ipywidgets)

Screen an equity index for the **top beats and misses** — actual reported EPS vs. the market
(consensus) estimate — using Yahoo Finance data via `yfinance`.

- Type the index in the **textbox** below (`SP500`, `NASDAQ100`, `DOW30`, or a comma-separated ticker list).
- Pick the **reporting quarter** (the calendar quarter in which earnings were announced).
- Click **Run screen** to get the top/bottom 5 EPS surprises and the **average surprise across all index members** that reported.

> A Streamlit version of the same screener lives in `streamlit_app.py`
> (`streamlit run earnings_screener/streamlit_app.py`). Both share the logic in `core.py`.


In [ ]:
# If needed, install dependencies first:
# %pip install yfinance ipywidgets pandas matplotlib lxml

import sys
from pathlib import Path

# Make core.py importable whether the notebook is opened from the repo root
# or from inside earnings_screener/.
here = Path.cwd()
for candidate in (here, here / "earnings_screener", here.parent):
    if (candidate / "core.py").exists():
        sys.path.insert(0, str(candidate))
        break

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from core import quarter_label, recent_quarters, run_screen

# Diverging pair from the reference palette: blue = beat, red = miss.
BEAT_COLOR, MISS_COLOR, INK = "#2a78d6", "#e34948", "#52514e"


## Controls

The **textbox** selects the index to analyse. Aliases like `S&P 500`, `^GSPC` or `SPX` all work.


In [ ]:
index_box = widgets.Text(
    value="SP500",
    description="Index:",
    placeholder="SP500, NASDAQ100, DOW30 or tickers e.g. AAPL, MSFT",
    layout=widgets.Layout(width="480px"),
)
quarter_dd = widgets.Dropdown(
    options=[(quarter_label(y, q), (y, q)) for y, q in recent_quarters(6)],
    description="Quarter:",
)
run_btn = widgets.Button(description="Run screen", button_style="primary", icon="search")
progress = widgets.IntProgress(value=0, min=0, max=1, description="Idle", layout=widgets.Layout(width="480px"))
out = widgets.Output()


def plot_extremes(result):
    extremes = pd.concat([result.top(5), result.bottom(5)]).drop_duplicates("symbol")
    extremes = extremes.sort_values("surprise_pct")
    colors = [BEAT_COLOR if v >= 0 else MISS_COLOR for v in extremes["surprise_pct"]]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    bars = ax.barh(extremes["symbol"], extremes["surprise_pct"], color=colors, height=0.55)
    ax.bar_label(bars, fmt="%+.1f%%", padding=3, color=INK, fontsize=9)
    ax.axvline(0, color="#c3c2b7", linewidth=1)
    ax.set_xlabel("EPS surprise vs. estimate (%)", color=INK)
    ax.set_title(f"Top 5 beats and bottom 5 misses — {result.label}", color=INK)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)
    ax.tick_params(colors=INK)
    ax.margins(x=0.15)
    fig.tight_layout()
    plt.show()


def display_tables(result):
    cols = {
        "symbol": "Ticker", "name": "Company", "earnings_date": "Reported on",
        "eps_estimate": "EPS estimate", "reported_eps": "Actual EPS", "surprise_pct": "Surprise %",
    }
    fmt = {"EPS estimate": "{:.2f}", "Actual EPS": "{:.2f}", "Surprise %": "{:+.1f}%"}
    print(f"\nTop 5 beats — {result.label}")
    display(result.top(5)[list(cols)].rename(columns=cols).style.format(fmt).hide(axis="index"))
    print(f"\nBottom 5 misses — {result.label}")
    display(result.bottom(5)[list(cols)].rename(columns=cols).style.format(fmt).hide(axis="index"))


def on_run(_):
    out.clear_output()
    year, quarter = quarter_dd.value

    def on_progress(done, total, symbol):
        progress.max, progress.value = total, done
        progress.description = f"{done}/{total}"

    with out:
        try:
            result = run_screen(index_box.value, year, quarter, progress=on_progress)
        except Exception as exc:
            print(f"Screen failed: {exc}")
            return
        finally:
            progress.description = "Done"

        scored = result.scored
        if scored.empty:
            print(f"No members of {index_box.value!r} reported in {result.label} yet — try the previous quarter.")
            return

        print(f"Average EPS surprise across {len(scored)} reported members ({result.label}): "
              f"{result.average_surprise:+.2f}%  (median {result.median_surprise:+.2f}%)")
        print(f"Beats: {result.beat_count}   Misses: {result.miss_count}   "
              f"Reported: {len(scored)} of {result.total_members}")
        if result.failed_symbols:
            print(f"Could not fetch {len(result.failed_symbols)} symbols "
                  f"(e.g. {', '.join(sorted(result.failed_symbols)[:5])})")
        display_tables(result)
        plot_extremes(result)


run_btn.on_click(on_run)
display(widgets.VBox([index_box, quarter_dd, widgets.HBox([run_btn, progress]), out]))


## Q: What were the top and bottom 5 EPS surprises for the S&P 500 this quarter?

Type `SP500` in the textbox above (the default) and click **Run screen** — or run the cell
below for a scripted answer. The screen pulls every S&P 500 member's latest earnings report
announced this calendar quarter, ranks the surprise % = (actual − estimate) / |estimate|,
and prints the five biggest beats, the five biggest misses, and the index-wide average.

*(Fetching ~500 tickers from Yahoo Finance takes a few minutes and needs internet access.)*


In [ ]:
result = run_screen("SP500")  # defaults to the current calendar quarter

if result.scored.empty:
    # Early in a quarter nothing has reported yet - fall back to the previous quarter.
    prev_year, prev_quarter = recent_quarters(2)[1]
    print(f"No S&P 500 earnings reported yet in {result.label}; "
          f"falling back to {quarter_label(prev_year, prev_quarter)}...")
    result = run_screen("SP500", prev_year, prev_quarter)

print(f"S&P 500 - {result.label}: {len(result.scored)} of {result.total_members} members reported")
print(f"Average EPS surprise: {result.average_surprise:+.2f}%   (median {result.median_surprise:+.2f}%)")
print(f"Beats: {result.beat_count}   Misses: {result.miss_count}")

cols = ["symbol", "name", "earnings_date", "eps_estimate", "reported_eps", "surprise_pct"]
print("\nTop 5 EPS beats:")
display(result.top(5)[cols])
print("\nBottom 5 EPS misses:")
display(result.bottom(5)[cols])
